# Scrapping the book review webpage

## 1. Import all libraries required

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt

## 2. Scrap one page

In [2]:
url = "https://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, "lxml")

books = soup.find_all("article", class_="product_pod")

data = []
for book in books:
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text.strip()
    availability = book.find("p", class_="instock availability").text.strip()
    rating = book.p["class"][1]

    data.append({
        "Title": title,
        "Price": price,
        "Availability": availability,
        "Rating": rating
    })

df = pd.DataFrame(data)
df.head()

,Title,Price,Availability,Rating
0,A Light in the Attic,Â£51.77,In stock,Three
1,Tipping the Velvet,Â£53.74,In stock,One
2,Soumission,Â£50.10,In stock,One
3,Sharp Objects,Â£47.82,In stock,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five


## 3. Scrap all 50 pages

In [3]:
base_url = "https://books.toscrape.com/catalogue/page-{}.html"
all_books = []

for page in range(1, 51):
    url = base_url.format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "lxml")

    books = soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text.strip()
        availability = book.find("p", class_="instock availability").text.strip()
        rating = book.p["class"][1]

        all_books.append({
            "Title": title,
            "Price": price,
            "Availability": availability,
            "Rating": rating
        })

    print(f"✅ Scraped page {page}")

✅ Scraped page 1
✅ Scraped page 2
✅ Scraped page 3
✅ Scraped page 4
✅ Scraped page 5
✅ Scraped page 6
✅ Scraped page 7
✅ Scraped page 8
✅ Scraped page 9
✅ Scraped page 10
✅ Scraped page 11
✅ Scraped page 12
✅ Scraped page 13
✅ Scraped page 14
✅ Scraped page 15
✅ Scraped page 16
✅ Scraped page 17
✅ Scraped page 18
✅ Scraped page 19
✅ Scraped page 20
✅ Scraped page 21
✅ Scraped page 22
✅ Scraped page 23
✅ Scraped page 24
✅ Scraped page 25
✅ Scraped page 26
✅ Scraped page 27
✅ Scraped page 28
✅ Scraped page 29
✅ Scraped page 30
✅ Scraped page 31
✅ Scraped page 32
✅ Scraped page 33
✅ Scraped page 34
✅ Scraped page 35
✅ Scraped page 36
✅ Scraped page 37
✅ Scraped page 38
✅ Scraped page 39
✅ Scraped page 40
✅ Scraped page 41
✅ Scraped page 42
✅ Scraped page 43
✅ Scraped page 44
✅ Scraped page 45
✅ Scraped page 46
✅ Scraped page 47
✅ Scraped page 48
✅ Scraped page 49
✅ Scraped page 50


## 4. Save and clean the data

In [6]:
# Re-scrape or re-decode correctly
response = requests.get(url)
response.encoding = "utf-8"  # force UTF-8 decoding
soup = BeautifulSoup(response.text, "html.parser")


In [7]:
import re
import unicodedata

def clean_price(p):
    # Normalize weird encodings (Â, Â, etc.)
    p = unicodedata.normalize("NFKD", p)
    # Remove all non-digit or dot characters
    p = re.sub(r"[^0-9.]", "", p)
    return float(p) if p else None

df["Price"] = df["Price"].apply(clean_price)


In [8]:
df["Price"].head(10)


0    51.77
1    53.74
2    50.10
3    47.82
4    54.23
5    22.65
6    33.34
7    17.93
8    22.60
9    52.15
Name: Price, dtype: float64

In [10]:
import unicodedata

# Clean up all text columns
for col in df.select_dtypes(include="object"):
    df[col] = df[col].apply(lambda x: unicodedata.normalize("NFKD", x).strip())

# Then clean Price
df["Price"] = df["Price"].str.replace("£", "").astype(float)

ValueError: could not convert string to float: 'Â51.77'

In [9]:
df = pd.DataFrame(all_books)

# Clean up
df["Price"] = df["Price"].str.replace("£", "").astype(float)
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["Rating"] = df["Rating"].map(rating_map)

# Save to CSV
df.to_csv("books_data.csv", index=False)
print("Saved to books_data.csv")

df.head()

ValueError: could not convert string to float: 'Â51.77'